# Laboratorium 5: Grafy wiedzy w zastosowaniach

Dziedzina: laureaci Nagrody Nobla urodzeni na terenie dzisiejszej Polski. Źródła: Wikidata (zapytania SPARQL), fragmenty haseł polskiej Wikipedii (licencja CC BY-SA 4.0), model `paraphrase-multilingual-MiniLM-L12-v2` i model językowy `Qwen/Qwen2.5-0.5B-Instruct` z Hugging Face.

In [ ]:
# W Colabie brakuje rdflib i owlrl; torch, transformers i spaCy są preinstalowane.
%pip install -q --disable-pip-version-check rdflib owlrl sentence-transformers

In [ ]:
import importlib, importlib.util, subprocess, sys

# Polski model spaCy (ok. 20 MB) pobieramy tylko wtedy, gdy go brakuje
if importlib.util.find_spec("pl_core_news_sm") is None:
    subprocess.run([sys.executable, "-m", "spacy", "download", "pl_core_news_sm"], check=True)
    importlib.invalidate_caches()

In [ ]:
import io, re, time, warnings
from collections import Counter, defaultdict

warnings.filterwarnings("ignore", message="IProgress not found")  # pasek postępu tqdm w Jupyterze bez ipywidgets

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
import owlrl, rdflib, spacy, torch, transformers
from rdflib import Graph, Literal, Namespace, URIRef, RDFS, XSD
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModelForCausalLM, AutoTokenizer

import huggingface_hub

RANDOM_STATE = 42
huggingface_hub.utils.logging.set_verbosity_error()
torch.manual_seed(RANDOM_STATE)
transformers.logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)
print("rdflib", rdflib.__version__, "| spaCy", spacy.__version__, "| torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
USER_AGENT = "MRW-lab/1.0 (zajecia dydaktyczne)"
WDQS = "https://query.wikidata.org/sparql"

WD = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")
P = Namespace("http://www.wikidata.org/prop/")
PS = Namespace("http://www.wikidata.org/prop/statement/")
PQ = Namespace("http://www.wikidata.org/prop/qualifier/")
EX = Namespace("http://example.org/nobel#")


def wikidata_csv(query, timeout=60):
    """Wykonuje zapytanie w Wikidata Query Service i zwraca DataFrame (kolumny jako tekst)."""
    for _ in range(3):
        r = requests.get(WDQS, params={"query": query},
                         headers={"User-Agent": USER_AGENT, "Accept": "text/csv"}, timeout=timeout)
        if r.status_code == 429:  # throttling: czekamy tyle, ile prosi serwer
            time.sleep(min(int(r.headers.get("Retry-After", 5)), 30))
            continue
        r.raise_for_status()
        return pd.read_csv(io.StringIO(r.text), dtype=str)
    raise RuntimeError("Wikidata: zbyt wiele zapytań (HTTP 429)")


def with_fallback(query, fallback_csv, normalize):
    """Zwraca wynik zapytania, a po błędzie sieci lub serwera dane zapasowe wbudowane w notebook."""
    try:
        return normalize(wikidata_csv(query)), "Wikidata"
    except Exception as e:
        print("Zapytanie nie powiodło się:", type(e).__name__, "(używam danych zapasowych)")
        return pd.read_csv(io.StringIO(fallback_csv), dtype=str), "dane zapasowe"


def qid(value):
    return value.rsplit("/", 1)[-1] if isinstance(value, str) else value

## Zadanie 1: Graf wiedzy z Wikidata

In [ ]:
# Sześć nagród: literatura, pokój, fizyka, chemia, fizjologia lub medycyna, ekonomia (Nagroda Banku Szwecji)
NOBEL = {"Q37922": "literatura", "Q35637": "pokój", "Q38104": "fizyka",
         "Q44585": "chemia", "Q80061": "medycyna", "Q47170": "ekonomia"}

Q_LAUREACI = """
SELECT ?osoba ?osobaLabel ?plec ?dataUr ?miejsce ?miejsceLabel ?nagroda ?nagrodaLabel ?rok WHERE {
  VALUES ?nagroda { %s }
  ?osoba p:P166 ?st . ?st ps:P166 ?nagroda .
  OPTIONAL { ?st pq:P585 ?rok }
  ?osoba wdt:P19 ?miejsce . ?miejsce wdt:P17 wd:Q36 .
  OPTIONAL { ?osoba wdt:P21 ?plec }
  OPTIONAL { ?osoba wdt:P569 ?dataUr }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "pl,en". }
} LIMIT 500
""" % " ".join("wd:" + q for q in NOBEL)


def normalize_laureaci(df):
    df = df.copy()
    for c in ["osoba", "miejsce", "nagroda"]:
        df[c] = df[c].map(qid)
    df["plec"] = df["plec"].map({"http://www.wikidata.org/entity/Q6581072": "K",
                                 "http://www.wikidata.org/entity/Q6581097": "M"})
    df["dataUr"] = df["dataUr"].str[:10]
    df["rok"] = df["rok"].str[:4]
    return df

In [ ]:
# Dane zapasowe: wyniki trzech zapytań z 23.10.2026, używane tylko wtedy, gdy Wikidata nie odpowiada
FALLBACK_LAUREACI = """
osoba,osobaLabel,plec,dataUr,miejsce,miejsceLabel,nagroda,nagrodaLabel,rok
Q127234,Albert Abraham Michelson,M,1852-12-19,Q1005414,Strzelno,Q38104,Nagroda Nobla w dziedzinie fizyki,1907
Q76425,Emil Adolf von Behring,M,1854-03-15,Q2800779,Ławice,Q80061,Nagroda Nobla w dziedzinie fizjologii lub medycyny,1901
Q76614,Friedrich Bergius,M,1884-10-11,Q9393436,Złotniki,Q44585,Nagroda Nobla w dziedzinie chemii,1931
Q57075,Fritz Haber,M,1868-12-09,Q1799,Wrocław,Q44585,Nagroda Nobla w dziedzinie chemii,1918
Q57192,Gerhard Domagk,M,1895-10-30,Q345189,Łagów,Q80061,Nagroda Nobla w dziedzinie fizjologii lub medycyny,1939
Q43523,Gerhart Hauptmann,M,1862-11-15,Q1001231,Szczawno-Zdrój,Q37922,Nagroda Nobla w dziedzinie literatury,1912
Q60108,Günter Blobel,M,1936-05-21,Q2369169,Niegosławice,Q80061,Nagroda Nobla w dziedzinie fizjologii lub medycyny,1999
Q41502,Henryk Sienkiewicz,M,1846-05-05,Q2750458,Wola Okrzejska,Q37922,Nagroda Nobla w dziedzinie literatury,1905
Q75612,Isaac Bashevis Singer,M,1904-07-14,Q2093413,Leoncin,Q37922,Nagroda Nobla w dziedzinie literatury,1978
Q75612,Isaac Bashevis Singer,M,1904-07-14,Q609246,Radzymin,Q37922,Nagroda Nobla w dziedzinie literatury,1978
Q165969,Isidor Isaac Rabi,M,1898-07-29,Q2588425,Rymanów,Q38104,Nagroda Nobla w dziedzinie fizyki,1944
Q219982,Joseph Rotblat,M,1908-11-04,Q270,Warszawa,Q35637,Pokojowa Nagroda Nobla,1995
Q71024,Klaus von Klitzing,M,1943-06-28,Q52903,Środa Wielkopolska,Q38104,Nagroda Nobla w dziedzinie fizyki,1985
Q35698,Konrad Bloch,M,1912-01-21,Q815721,Nysa,Q80061,Nagroda Nobla w dziedzinie fizjologii lub medycyny,1964
Q76595,Kurt Alder,M,1902-07-10,Q104302,Chorzów,Q44585,Nagroda Nobla w dziedzinie chemii,1950
Q444,Lech Wałęsa,M,1943-09-29,Q4700601,Popowo,Q35637,Pokojowa Nagroda Nobla,1983
Q57100,Maria Goeppert-Mayer,K,1906-06-28,Q588,Katowice,Q38104,Nagroda Nobla w dziedzinie fizyki,1963
Q7186,Maria Skłodowska-Curie,K,1867-11-07,Q270,Warszawa,Q38104,Nagroda Nobla w dziedzinie fizyki,1903
Q7186,Maria Skłodowska-Curie,K,1867-11-07,Q9365159,Ulica Freta w Warszawie,Q38104,Nagroda Nobla w dziedzinie fizyki,1903
Q7186,Maria Skłodowska-Curie,K,1867-11-07,Q270,Warszawa,Q44585,Nagroda Nobla w dziedzinie chemii,1911
Q7186,Maria Skłodowska-Curie,K,1867-11-07,Q9365159,Ulica Freta w Warszawie,Q44585,Nagroda Nobla w dziedzinie chemii,1911
Q58978,Max Born,M,1882-12-11,Q1799,Wrocław,Q38104,Nagroda Nobla w dziedzinie fizyki,1954
Q254032,Olga Tokarczuk,K,1962-01-29,Q457865,Sulechów,Q37922,Nagroda Nobla w dziedzinie literatury,2018
Q57072,Otto Stern,M,1888-02-17,Q393171,Q393171,Q38104,Nagroda Nobla w dziedzinie fizyki,1943
Q57089,Paul Ehrlich,M,1854-03-14,Q719954,Strzelin,Q80061,Nagroda Nobla w dziedzinie fizjologii lub medycyny,1908
Q12830,Reinhard Selten,M,1930-10-05,Q1799,Wrocław,Q47170,Nagroda Banku Szwecji im. Alfreda Nobla w dziedzinie ekonomii,1994
Q115490,Tadeusz Reichstein,M,1897-07-20,Q106681,Włocławek,Q80061,Nagroda Nobla w dziedzinie fizjologii lub medycyny,1950
Q57125,Walther Nernst,M,1864-06-25,Q324912,Wąbrzeźno,Q44585,Nagroda Nobla w dziedzinie chemii,1920
Q42552,Wisława Szymborska,K,1923-07-02,Q11828000,Prowent,Q37922,Nagroda Nobla w dziedzinie literatury,1996
Q121180,Władysław Reymont,M,1867-05-07,Q2248969,Kobiele Wielkie,Q37922,Nagroda Nobla w dziedzinie literatury,1924
"""

FALLBACK_UCZELNIE = """
osoba,uczelnia,uczelniaLabel
Q115490,Q11942,Politechnika Federalna w Zurychu
Q127234,Q152087,Uniwersytet Humboldtów w Berlinie
Q127234,Q559549,United States Naval Academy
Q12830,Q50662,Uniwersytet Johanna Wolfganga Goethego we Frankfurcie nad Menem
Q165969,Q12060888,John Jay Educational Campus
Q165969,Q49088,Uniwersytet Columbia
Q165969,Q49115,Cornell University
Q219982,Q144488,Uniwersytet Warszawski
Q219982,Q170027,University of London
Q219982,Q499510,University of Liverpool
Q254032,Q9379679,Wydział Psychologii Uniwersytetu Warszawskiego
Q35698,Q157808,Uniwersytet Techniczny w Monachium
Q35698,Q49088,Uniwersytet Columbia
Q35698,Q55044,Uniwersytet Ludwika i Maksymiliana w Monachium
Q41502,Q144488,Uniwersytet Warszawski
Q42552,Q189441,Uniwersytet Jagielloński w Krakowie
Q43523,Q154561,Uniwersytet w Jenie
Q57072,Q1698809,Johanneum
Q57072,Q315658,Uniwersytet Wrocławski
Q57072,Q50662,Uniwersytet Johanna Wolfganga Goethego we Frankfurcie nad Menem
Q57075,Q11942,Politechnika Federalna w Zurychu
Q57075,Q1329669,Gimnazjum św. Elżbiety
Q57075,Q151510,Uniwersytet w Heidelbergu
Q57075,Q152087,Uniwersytet Humboldtów w Berlinie
Q57075,Q1698809,Johanneum
Q57075,Q51985,Berliński Uniwersytet Techniczny
Q57089,Q1297729,Gimnazjum św. Marii Magdaleny
Q57089,Q153987,Uniwersytet we Fryburgu
Q57089,Q154804,Uniwersytet w Lipsku
Q57089,Q157575,Uniwersytet w Strasburgu
Q57089,Q159895,Uniwersytet w Rostocku
Q57089,Q315658,Uniwersytet Wrocławski
Q57100,Q152838,Uniwersytet w Getyndze
Q57125,Q152087,Uniwersytet Humboldtów w Berlinie
Q57125,Q154804,Uniwersytet w Lipsku
Q57125,Q161976,Uniwersytet w Würzburgu
Q57125,Q206702,Uniwersytet w Zurychu
Q57125,Q622683,Uniwersytet Karola i Franciszka w Grazu
Q57192,Q156737,Uniwersytet Christiana-Albrechta w Kilonii
Q58978,Q151510,Uniwersytet w Heidelbergu
Q58978,Q152838,Uniwersytet w Getyndze
Q58978,Q206702,Uniwersytet w Zurychu
Q58978,Q315658,Uniwersytet Wrocławski
Q58978,Q863940,Gonville and Caius College
Q60108,Q153978,Uniwersytet w Tybindze
Q60108,Q153987,Uniwersytet we Fryburgu
Q60108,Q156737,Uniwersytet Christiana-Albrechta w Kilonii
Q60108,Q50662,Uniwersytet Johanna Wolfganga Goethego we Frankfurcie nad Menem
Q60108,Q55044,Uniwersytet Ludwika i Maksymiliana w Monachium
Q60108,Q838330,University of Wisconsin-Madison
Q71024,Q157808,Uniwersytet Techniczny w Monachium
Q71024,Q161976,Uniwersytet w Würzburgu
Q71024,Q713563,Artland-Gymnasium
Q71024,Q734324,Uniwersytet Techniczny w Brunszwiku
Q7186,Q1048461,Uniwersytet Latający
Q7186,Q209842,Uniwersytet Paryski
Q7186,Q3064332,Faculté des sciences de Paris
Q7186,Q9383191,X Liceum Ogólnokształcące im. Królowej Jadwigi w Warszawie
Q76425,Q152087,Uniwersytet Humboldtów w Berlinie
Q76595,Q152087,Uniwersytet Humboldtów w Berlinie
Q76595,Q156737,Uniwersytet Christiana-Albrechta w Kilonii
Q76614,Q154804,Uniwersytet w Lipsku
Q76614,Q315658,Uniwersytet Wrocławski
"""

FALLBACK_JEDNOSTKI = """
x,xLabel,y,yLabel
Q1001231,Szczawno-Zdrój,Q1803526,Powiat Waldenburg
Q1001231,Szczawno-Zdrój,Q280091,powiat wałbrzyski
Q1005414,Strzelno,Q2347468,Strzelno
Q101575,powiat poznański,Q54187,województwo wielkopolskie
Q1022950,Niegosławice,Q318442,powiat żagański
Q104302,Chorzów,Q54181,województwo śląskie
Q106681,Włocławek,Q54153,województwo kujawsko-pomorskie
Q1135390,powiat nowodworski,Q54169,województwo mazowieckie
Q1135409,powiat iławski,Q54184,województwo warmińsko-mazurskie
Q1135416,powiat radomszczański,Q20033850,województwo łódzkie
Q1135416,powiat radomszczański,Q54158,województwo łódzkie
Q1149128,powiat krośnieński,Q54175,województwo podkarpackie
Q11825638,Powiat krośnieński (Galicja),Q2396442,Królestwo Galicji i Lodomerii
Q11825720,powiat radzymiński,Q2520566,gubernia warszawska
Q11825720,powiat radzymiński,Q4163574,województwo warszawskie
Q11825741,powiat sanocki,Q2303972,województwo lwowskie
Q11825741,powiat sanocki,Q2396442,Królestwo Galicji i Lodomerii
Q11828000,Prowent,Q51408,Kórnik
Q1309994,Powiat Neisse,Q827092,rejencja opolska
Q133280,powiat średzki,Q54187,województwo wielkopolskie
Q1475365,Powiat Graudenz,Q479818,Rejencja kwidzyńska
Q161036,Wolne Państwo Prusy,Q1206012,Rzesza Niemiecka
Q161036,Wolne Państwo Prusy,Q55300,radziecka strefa okupacyjna w Niemczech
Q161036,Wolne Państwo Prusy,Q55304,amerykańska strefa okupacyjna w Niemczech
Q161036,Wolne Państwo Prusy,Q55306,brytyjska strefa okupacyjna w Niemczech
Q161036,Wolne Państwo Prusy,Q55309,francuska strefa okupacyjna w Niemczech
Q161947,Prusy Zachodnie,Q161036,Wolne Państwo Prusy
Q161947,Prusy Zachodnie,Q27306,Królestwo Prus
Q1799,Wrocław,Q54150,województwo dolnośląskie
Q179986,powiat strzeliński,Q54150,województwo dolnośląskie
Q1803486,Powiat Strehlen,Q675032,Śląsk
Q1803526,Powiat Waldenburg,Q675032,Śląsk
Q1838850,Krzywda,Q914615,powiat łukowski
Q1974958,Sulechów,Q318384,powiat zielonogórski
Q20033850,województwo łódzkie,Q211274,Polska Rzeczpospolita Ludowa
Q205816,Rejencja legnicka,Q675032,Śląsk
Q205816,Rejencja legnicka,Q700812,Dolny Śląsk
Q2093413,Leoncin,Q2123929,Leoncin
Q2117812,Strzelin,Q179986,powiat strzeliński
Q2123929,Leoncin,Q1135390,powiat nowodworski
Q21843741,Ramosky Uyezd,Q2532795,gubernia radomska
Q221457,Królestwo Polskie,Q34266,Imperium Rosyjskie
Q2228704,Nysa,Q461537,powiat nyski
Q2248969,Kobiele Wielkie,Q21843741,Ramosky Uyezd
Q2248969,Kobiele Wielkie,Q2460470,Kobiele Wielkie
Q2275755,Radzymin,Q936180,powiat wołomiński
Q2303972,województwo lwowskie,Q207272,II Rzeczpospolita
Q2347468,Strzelno,Q604053,powiat mogileński
Q2369169,Niegosławice,Q1022950,Niegosławice
Q2369169,Niegosławice,Q883882,Powiat Sprottau
Q2396442,Królestwo Galicji i Lodomerii,Q131964,Cesarstwo Austrii
Q2396442,Królestwo Galicji i Lodomerii,Q533534,Przedlitawia
Q2460470,Kobiele Wielkie,Q1135416,powiat radomszczański
Q2460708,Łagów,Q318397,powiat świebodziński
Q2520566,gubernia warszawska,Q221457,Królestwo Polskie
Q2520566,gubernia warszawska,Q4104100,Generał-Gubernatorstwo Warszawskie
Q2532795,gubernia radomska,Q221457,Królestwo Polskie
Q2532795,gubernia radomska,Q4104100,Generał-Gubernatorstwo Warszawskie
Q2588302,województwo włocławskie,Q211274,Polska Rzeczpospolita Ludowa
Q2588302,województwo włocławskie,Q36,Polska
Q2588425,Rymanów,Q655124,Rymanów
Q270,Warszawa,Q54169,województwo mazowieckie
Q2750458,Wola Okrzejska,Q1838850,Krzywda
Q2800779,Ławice,Q555266,Iława
Q280091,powiat wałbrzyski,Q54150,województwo dolnośląskie
Q318384,powiat zielonogórski,Q54157,województwo lubuskie
Q318397,powiat świebodziński,Q54157,województwo lubuskie
Q318442,powiat żagański,Q54157,województwo lubuskie
Q324912,Wąbrzeźno,Q1475365,Powiat Graudenz
Q324912,Wąbrzeźno,Q1787234,Powiat Briesen
Q324912,Wąbrzeźno,Q647959,Ryńsk
Q345189,Łagów,Q2460708,Łagów
Q393171,Q393171,Q54181,województwo śląskie
Q4104100,Generał-Gubernatorstwo Warszawskie,Q34266,Imperium Rosyjskie
Q4163574,województwo warszawskie,Q211274,Polska Rzeczpospolita Ludowa
Q457865,Sulechów,Q1974958,Sulechów
Q461537,powiat nyski,Q54171,województwo opolskie
Q4700601,Popowo,Q618404,Tłuchowo
Q479818,Rejencja kwidzyńska,Q161947,Prusy Zachodnie
Q479818,Rejencja kwidzyńska,Q675291,Prusy
Q51408,Kórnik,Q555812,Kórnik
Q52903,Środa Wielkopolska,Q554658,Środa Wielkopolska
Q533534,Przedlitawia,Q28513,Austro-Węgry
Q54150,województwo dolnośląskie,Q36,Polska
Q54153,województwo kujawsko-pomorskie,Q36,Polska
Q54155,województwo lubelskie,Q36,Polska
Q54157,województwo lubuskie,Q36,Polska
Q54158,województwo łódzkie,Q36,Polska
Q54169,województwo mazowieckie,Q36,Polska
Q54171,województwo opolskie,Q36,Polska
Q54175,województwo podkarpackie,Q36,Polska
Q54181,województwo śląskie,Q36,Polska
Q54184,województwo warmińsko-mazurskie,Q36,Polska
Q54187,województwo wielkopolskie,Q36,Polska
Q554658,Środa Wielkopolska,Q133280,powiat średzki
Q555266,Iława,Q1135409,powiat iławski
Q555812,Kórnik,Q101575,powiat poznański
Q588,Katowice,Q54181,województwo śląskie
Q604053,powiat mogileński,Q54153,województwo kujawsko-pomorskie
Q609246,Radzymin,Q11825720,powiat radzymiński
Q609246,Radzymin,Q2275755,Radzymin
Q618404,Tłuchowo,Q773421,powiat lipnowski
Q647959,Ryńsk,Q774738,powiat wąbrzeski
Q655124,Rymanów,Q1149128,powiat krośnieński
Q655124,Rymanów,Q11825638,Powiat krośnieński (Galicja)
Q655124,Rymanów,Q11825741,powiat sanocki
Q655124,Rymanów,Q1422856,województwo krośnieńskie
Q675032,Śląsk,Q27306,Królestwo Prus
Q675291,Prusy,Q27306,Królestwo Prus
Q700812,Dolny Śląsk,Q161036,Wolne Państwo Prusy
Q704495,Górny Śląsk,Q161036,Wolne Państwo Prusy
Q7095,Śródmieście,Q270,Warszawa
Q719954,Strzelin,Q1803486,Powiat Strehlen
Q719954,Strzelin,Q2117812,Strzelin
Q773421,powiat lipnowski,Q2588302,województwo włocławskie
Q773421,powiat lipnowski,Q54153,województwo kujawsko-pomorskie
Q774738,powiat wąbrzeski,Q54153,województwo kujawsko-pomorskie
Q815721,Nysa,Q1309994,Powiat Neisse
Q815721,Nysa,Q2228704,Nysa
Q827092,rejencja opolska,Q675032,Śląsk
Q827092,rejencja opolska,Q704495,Górny Śląsk
Q883882,Powiat Sprottau,Q205816,Rejencja legnicka
Q914615,powiat łukowski,Q54155,województwo lubelskie
Q936180,powiat wołomiński,Q54169,województwo mazowieckie
Q9365159,Ulica Freta w Warszawie,Q270,Warszawa
Q9365159,Ulica Freta w Warszawie,Q7095,Śródmieście
Q9393436,Złotniki,Q1799,Wrocław
"""

In [ ]:
# Pobierz laureatów zapytaniem Q_LAUREACI (z danymi zapasowymi), a następnie napisz dwa zapytania:
# Q_UCZELNIE (wdt:P69 dla znalezionych osób) i Q_JEDNOSTKI (łańcuch wdt:P131 w górę od miejsc urodzenia).
# Do Q_JEDNOSTKI dodaj: hint:Query hint:optimizer "None" .
# Między zapytaniami odczekaj sekundę. Wypisz liczbę wierszy, osób i miejsc urodzenia.

In [ ]:
# Zbuduj graf rdflib: etykiety rdfs:label@pl, krawędzie wdt:P19, wdt:P166, wdt:P69, wdt:P131,
# płeć (ex:plec), datę urodzenia (wdt:P569) oraz wypowiedzi p:P166 / ps:P166 / pq:P585 z rokiem nagrody.
# Wypisz liczbę trójek i liczności predykatów.

In [ ]:
# Przenieś krawędzie między encjami Wikidata do nx.MultiDiGraph.
# Policz węzły, krawędzie, gęstość, słabo spójne składowe, statystyki stopni,
# pokaż 8 węzłów o największym stopniu i narysuj rozkład stopni.

In [ ]:
# Sprawdź jakość danych zapytaniami SPARQL na lokalnym grafie:
# osoby z więcej niż jednym miejscem urodzenia, osoby z więcej niż jedną nagrodą,
# węzły bez etykiety (etykieta równa Q-identyfikatorowi), laureaci bez informacji o uczelni.

(uzupełnij)

## Zadanie 2: Ekstrakcja trójek z tekstu i łączenie encji

In [ ]:
# Początkowe zdania haseł polskiej Wikipedii (stan z października 2026), licencja CC BY-SA 4.0,
# autorzy: zob. historia edycji każdego hasła. Skróty oznaczono [...], myślniki ujednolicono do "-".
TEXTS = {
    "Maria Skłodowska-Curie": "Maria Salomea Skłodowska-Curie (ur. 7 listopada 1867 w Warszawie, zm. 4 lipca 1934 w Passy) - polsko-francuska uczona w dziedzinach fizyki doświadczalnej i chemii fizycznej, podwójna noblistka - laureatka Nagrody Nobla z fizyki (1903) i chemii (1911).",
    "Henryk Sienkiewicz": "Henryk Adam Aleksander Pius Sienkiewicz, ps. „Litwos”, „Musagetes”, „Juliusz Polkowski”, „K. Dobrzyński” (ur. 5 maja 1846 w Woli Okrzejskiej, zm. 15 listopada 1916 w Vevey) - polski nowelista, powieściopisarz epoki pozytywizmu i publicysta; laureat Nagrody Nobla w dziedzinie literatury (1905) za całokształt twórczości, jeden z najpopularniejszych polskich pisarzy przełomu XIX i XX w.",
    "Władysław Reymont": "Władysław Stanisław Reymont, właśc. Stanisław Władysław Rejment (ur. 7 maja 1867 w Kobielach Wielkich, zm. 5 grudnia 1925 w Warszawie) - polski pisarz, prozaik i nowelista. Laureat Nagrody Nobla w dziedzinie literatury w 1924 za czterotomową „epopeję chłopską” Chłopi.",
    "Czesław Miłosz": "Czesław Miłosz (ur. 30 czerwca 1911 w Szetejniach, zm. 14 sierpnia 2004 w Krakowie) - polski poeta, prozaik, eseista, historyk literatury, tłumacz, dyplomata. Laureat Nagrody Nobla w dziedzinie literatury (1980); profesor Uniwersytetu Kalifornijskiego w Berkeley.",
    "Wisława Szymborska": "Wisława Maria Szymborska-Włodek (ur. 2 lipca 1923 w Prowencie, obecnie część Kórnika, zm. 1 lutego 2012 w Krakowie) - polska poetka, eseistka, krytyczka, tłumaczka, felietonistka; laureatka Nagrody Nobla w dziedzinie literatury (1996), dama Orderu Orła Białego.",
    "Lech Wałęsa": "Lech Wałęsa (ur. 29 września 1943 w Popowie) - polski robotnik, działacz związkowy i polityk, Prezydent Rzeczypospolitej Polskiej w latach 1990-1995. [...] Laureat Pokojowej Nagrody Nobla w 1983.",
    "Olga Tokarczuk": "Olga Nawoja Tokarczuk (ur. 29 stycznia 1962 w Sulechowie) - polska pisarka, eseistka, scenarzystka i psycholożka. Jest laureatką Nagrody Nobla w dziedzinie literatury za rok 2018 i zdobywczynią Międzynarodowej Nagrody Bookera (2018) za powieść „Bieguni”.",
    "Tadeusz Reichstein": "Tadeusz Reichstein, Tadeus Reichstein (ur. 20 lipca 1897 we Włocławku, zm. 1 sierpnia 1996 w Bazylei) - szwajcarski biochemik pochodzenia polsko-żydowskiego, laureat Nagrody Nobla w dziedzinie fizjologii lub medycyny w 1950 r.",
    "Józef Rotblat": "Józef Rotblat, również Joseph Rotblat (ur. 4 listopada 1908 w Warszawie, zm. 31 sierpnia 2005 w Londynie) - polsko-brytyjski fizyk pochodzenia żydowskiego, radiobiolog, do 1944 uczestnik projektu Manhattan. Założyciel i lider pacyfistycznego ruchu naukowców Pugwash, laureat Pokojowej Nagrody Nobla (1995).",
    "Isaac Bashevis Singer": "Isaac Bashevis Singer ([...] ur. prawdopodobnie 21 listopada 1902 w Leoncinie, zm. 24 lipca 1991 w Surfside w stanie Floryda) - amerykański pisarz będący z pochodzenia polskim Żydem, tworzący głównie w języku jidysz, dziennikarz. Laureat Nagrody Nobla w dziedzinie literatury za 1978 rok.",
}

# Ręcznie oznaczone trójki (podmiot = tytuł hasła, miejsca w mianowniku)
GOLD = {
    ("Maria Skłodowska-Curie", "urodzonyW", "Warszawa"), ("Maria Skłodowska-Curie", "zmarlW", "Passy"),
    ("Maria Skłodowska-Curie", "rokUrodzenia", "1867"), ("Maria Skłodowska-Curie", "laureat", "fizyka"),
    ("Maria Skłodowska-Curie", "laureat", "chemia"),
    ("Henryk Sienkiewicz", "urodzonyW", "Wola Okrzejska"), ("Henryk Sienkiewicz", "zmarlW", "Vevey"),
    ("Henryk Sienkiewicz", "rokUrodzenia", "1846"), ("Henryk Sienkiewicz", "laureat", "literatura"),
    ("Władysław Reymont", "urodzonyW", "Kobiele Wielkie"), ("Władysław Reymont", "zmarlW", "Warszawa"),
    ("Władysław Reymont", "rokUrodzenia", "1867"), ("Władysław Reymont", "laureat", "literatura"),
    ("Czesław Miłosz", "urodzonyW", "Szetejnie"), ("Czesław Miłosz", "zmarlW", "Kraków"),
    ("Czesław Miłosz", "rokUrodzenia", "1911"), ("Czesław Miłosz", "laureat", "literatura"),
    ("Wisława Szymborska", "urodzonyW", "Prowent"), ("Wisława Szymborska", "zmarlW", "Kraków"),
    ("Wisława Szymborska", "rokUrodzenia", "1923"), ("Wisława Szymborska", "laureat", "literatura"),
    ("Lech Wałęsa", "urodzonyW", "Popowo"), ("Lech Wałęsa", "rokUrodzenia", "1943"),
    ("Lech Wałęsa", "laureat", "pokój"),
    ("Olga Tokarczuk", "urodzonyW", "Sulechów"), ("Olga Tokarczuk", "rokUrodzenia", "1962"),
    ("Olga Tokarczuk", "laureat", "literatura"),
    ("Tadeusz Reichstein", "urodzonyW", "Włocławek"), ("Tadeusz Reichstein", "zmarlW", "Bazylea"),
    ("Tadeusz Reichstein", "rokUrodzenia", "1897"), ("Tadeusz Reichstein", "laureat", "medycyna"),
    ("Józef Rotblat", "urodzonyW", "Warszawa"), ("Józef Rotblat", "zmarlW", "Londyn"),
    ("Józef Rotblat", "rokUrodzenia", "1908"), ("Józef Rotblat", "laureat", "pokój"),
    ("Isaac Bashevis Singer", "urodzonyW", "Leoncin"), ("Isaac Bashevis Singer", "zmarlW", "Surfside"),
    ("Isaac Bashevis Singer", "rokUrodzenia", "1902"), ("Isaac Bashevis Singer", "laureat", "literatura"),
}

# Wzorzec do łączenia encji: Q-identyfikator osoby i zbiór miejsc urodzenia (wdt:P19) w Wikidata
EL_GOLD = {
    "Maria Skłodowska-Curie": ("Q7186", {"Q270", "Q9365159"}),
    "Henryk Sienkiewicz": ("Q41502", {"Q2750458"}),
    "Władysław Reymont": ("Q121180", {"Q2248969"}),
    "Czesław Miłosz": ("Q45970", {"Q391479", "Q134712"}),
    "Wisława Szymborska": ("Q42552", {"Q11828000"}),
    "Lech Wałęsa": ("Q444", {"Q4700601"}),
    "Olga Tokarczuk": ("Q254032", {"Q457865"}),
    "Tadeusz Reichstein": ("Q115490", {"Q106681"}),
    "Józef Rotblat": ("Q219982", {"Q270"}),
    "Isaac Bashevis Singer": ("Q75612", {"Q2093413", "Q609246"}),
}
print(len(TEXTS), "tekstów,", len(GOLD), "trójek wzorcowych")

In [ ]:
# (opcjonalnie) Jednym zapytaniem do API MediaWiki (action=query, prop=revisions) pobierz numery
# bieżących rewizji haseł z TEXTS. Do czego przydają się przy ponownym wykorzystaniu tekstu na licencji CC BY-SA?

In [ ]:
# Napisz ekstraktor trójek (urodzonyW, zmarlW, rokUrodzenia, laureat) oparty na wyrażeniach regularnych.
# Nazwy miejsc sprowadź do mianownika lematyzatorem spaCy (pl_core_news_sm); analizuj całe zdanie, nie samą wzmiankę.
# Przygotuj dwa warianty: bez lematyzacji i z lematyzacją. Wypisz trójki spoza GOLD.

In [ ]:
# Policz precyzję, pełność i F1 (ogółem i dla każdej relacji) dla obu wariantów ekstraktora.
# Wypisz trójki pominięte (FN) i błędne (FP), nazwij przyczynę każdego błędu.

In [ ]:
# Popraw ekstraktor tak, żeby obsłużył konstrukcję współrzędną "z fizyki (1903) i chemii (1911)".
# Ponownie policz miary. Czy ręczny słownik poprawek lematów jest dobrym rozwiązaniem?

In [ ]:
# Kandydaci z Wikidata dla etykiet używanych w łączeniu encji (zapis z 23.10.2026), na wypadek awarii sieci
FALLBACK_EL = """
label,item,opis,linki,czlowiek
Henryk Sienkiewicz,Q27865451,badmintonista,0,true
Henryk Sienkiewicz,Q41502,"polski pisarz i publicysta, noblista literacki",139,true
Henryk Sienkiewicz,Q43982475,"żołnierz Legii Cudzoziemskiej, stryjeczny brat pisarza o tym samym imieniu i nazwisku",1,true
Henryk Sienkiewicz,Q9288700,polski polityk,2,true
Isaac Bashevis Singer,Q75612,"pisarz amerykański, noblista",113,true
Józef Rotblat,Q219982,polsko-brytyjski fizyk pochodzenia żydowskiego (1908-2005),61,true
Kobiele Wielkie,Q2248969,wieś w województwie łódzkim,14,false
Kobiele Wielkie,Q2460470,gmina wiejska w województwie łódzkim,17,false
Kobiele Wielkie,Q9308676,leśny rezerwat przyrody w gminie Kobiele Wielkie  w woj. łódzkim,3,false
Lech Wałęsa,Q444,"polski polityk, 2. prezydent III RP w latach 1990-1995",176,true
Leoncin,Q20660511,strona ujednoznaczniająca w projekcie Wikimedia,1,false
Leoncin,Q2093413,wieś w województwie mazowieckim,16,false
Leoncin,Q2123929,gmina wiejska w województwie mazowieckim,17,false
Maria Skłodowska-Curie,Q132188977,rzeźba przed Państwowym Instytutem Badawczym w Gliwicach z 1994 roku,1,false
Maria Skłodowska-Curie,Q26963166,film w reżyserii Marie-Noëlle Sehr,10,false
Maria Skłodowska-Curie,Q7186,"polsko-francuska uczona, fizyczka (1867-1934)",248,true
Olga Tokarczuk,Q254032,polska pisarka i psycholożka,102,true
Popowo,Q11823945,,4,false
Popowo,Q1214625,strona ujednoznaczniająca,11,false
Popowo,Q123463320,jezioro,1,false
Popowo,Q16590183,przystanek kolejowy w województwie pomorskim,2,false
Popowo,Q2746191,"wieś w województwie kujawsko-pomorskim, powiecie lipnowskim, gminie Lipno",13,false
Popowo,Q406068,miasto w Bułgarii,27,false
Popowo,Q4372806,"wieś w Bułgarii, w obwodzie Jamboł",11,false
Popowo,Q4700601,"wieś w Polsce, w województwie kujawsko-pomorskim",15,false
Popowo,Q4977437,wieś w województwie pomorskim,10,false
Popowo,Q56328638,część wsi Ławica w Polsce,1,false
Popowo,Q5753463,"wieś w Polsce, w województwie podlaskim",11,false
Popowo,Q7229485,"wieś w województwie zachodniopomorskim, powiecie koszalińskim",8,false
Popowo,Q7229486,"wieś w województwie wielkopolskim, powiecie międzychodzkim",7,false
Popowo,Q7229487,"wieś w województwie lubuskim, powiecie międzyrzeckim",8,false
Popowo,Q7229488,"wieś w województwie lubuskim, powiecie nowosolskim",8,false
Popowo,Q7229490,"wieś w województwie pomorskim, powiecie nowodworskim",5,false
Popowo,Q7229491,"wieś w województwie wielkopolskim, powiecie obornickim",7,false
Popowo,Q7229492,"wieś w województwie wielkopolskim, powiecie szamotulskim",8,false
Popowo,Q7229493,wieś w województwie warmińsko-mazurskim,8,false
Popowo,Q7229495,"wieś w województwie zachodniopomorskim, powiecie wałeckim",5,false
Popowo,Q784365,wieś w województwie podlaskim,11,false
Prowent,Q11827999,strona ujednoznaczniająca,1,false
Prowent,Q11828000,część miasta Kórnik,1,false
Sulechów,Q16607915,strona ujednoznaczniająca,2,false
Sulechów,Q1974958,gmina miejsko-wiejska w województwie lubuskim,16,false
Sulechów,Q2936786,stacja kolejowa,3,false
Sulechów,Q457865,miasto w województwie lubuskim,39,false
Sulechów,Q7636106,wieś w województwie małopolskim,9,false
Szetejnie,Q391479,"wieś na Litwie, nad Niewiażą, w okręgu kowieńskim",15,false
Tadeusz Reichstein,Q115490,"biochemik szwajcarski pochodzenia polskiego, noblista",69,true
Warszawa,Q104637651,obraz Eugeniusza Eibischa,0,false
Warszawa,Q104648763,obraz Edward Kokoszko,0,false
Warszawa,Q109008488,definicja w „Słowniku etymologicznym języka polskiego”,1,false
Warszawa,Q109020651,wiersz Marii Konopnickiej,1,false
Warszawa,Q109251154,"definicja w „Słowniku prawdy i zdrowego rozsądku"" Kazimierza Bartoszewicza, wyd. 1905",1,false
Warszawa,Q1147772,album zespołu Porcupine Tree,4,false
Warszawa,Q118077231,nazwisko,0,false
Warszawa,Q126324918,singel zespołu T.Love,1,false
Warszawa,Q136437909,wiersz Krzysztofa Kamila Baczyńskiego,0,false
Warszawa,Q138769710,powieść Mariana Gawalewicza z 1901,0,false
Warszawa,Q19172583,wiersz  Artura Bartelsa,1,false
Warszawa,Q19172585,wiersz Kazimierza Laskowskiego,1,false
Warszawa,Q19172589,wiersz Kazimierza Perzyńskiego,1,false
Warszawa,Q19172593,wiersz  Władysława Syrokomli,1,false
Warszawa,Q19172597,wiersz Jerzego Żuławskiego,1,false
Warszawa,Q24729120,wieś w Rosji,9,false
Warszawa,Q270,"stolica Polski i jej najludniejsze miasto, stolica województwa mazowieckiego",288,false
Warszawa,Q30910818,wieś w gminie Krypno,2,false
Warszawa,Q30911311,część wsi Bondyrz,2,false
Warszawa,Q30911357,część wsi Feliksówka,2,false
Warszawa,Q30911370,wieś w gminie Krasnobród,2,false
Warszawa,Q404516,strona ujednoznaczniająca,14,false
Warszawa,Q48847697,część wsi Hłudno,1,false
Warszawa,Q49631213,miasto Warszawa traktowane jak powiat,1,false
Warszawa,Q546524,utwór Davida Bowiego,10,false
Warszawa,Q65239227,,1,false
Warszawa,Q7970959,"wieś w województwie pomorskim, powiecie kościerskim",4,false
Warszawa,Q7970960,album zespołu Praxis,2,false
Warszawa,Q9371127,film Dariusza Gajewskiego z 2003,1,false
Warszawa,Q9371129,"projekt wydzielonego miasta stołecznego, posiadającego prawa województwa",1,false
Warszawa,Q9371134,wieś w województwie wielkopolskim,3,false
Wisława Szymborska,Q42552,"polska poetka, noblistka",122,true
Wola Okrzejska,Q2750458,wieś w województwie lubelskim,21,false
Władysław Reymont,Q121180,"polski pisarz epoki Młodej Polski, noblista literacki (1867-1925)",108,true
Władysław Reymont,Q50385761,strona ujednoznaczniająca w projekcie Wikimedia,1,false
"""

In [ ]:
# Wygeneruj kandydatów do łączenia encji JEDNYM zapytaniem SPARQL: VALUES z etykietami
# (tytuły haseł i wydobyte miejsca urodzenia), dopasowanie rdfs:label lub skos:altLabel,
# liczba linków do Wikipedii (wikibase:sitelinks), opis po polsku i to, czy kandydat jest człowiekiem (wdt:P31 wd:Q5).
# Użyj with_fallback z FALLBACK_EL.

In [ ]:
# Zaimplementuj dwie strategie wyboru kandydata: (a) największa liczba linków (popularność),
# (b) popularność po zawężeniu kontekstem: dla osób tylko ludzie, dla miejsc tylko kandydaci obecni w grafie z zadania 1.
# Policz trafność dla osób i miejsc urodzenia względem EL_GOLD i pokaż przypadki błędne.

In [ ]:
# Połącz osoby przez sitelink do polskiej Wikipedii (schema:about, schema:isPartOf, schema:name) i porównaj
# z łączeniem po etykiecie. Dla której osoby łączenie po etykiecie zawiodło i dlaczego?

(uzupełnij)

## Zadanie 3: Odpowiadanie na pytania z grafu

In [ ]:
# Zestaw pytań używany w zadaniach 3, 4 i 5. "jedna": wystarczy jedna z odpowiedzi; "lista": potrzebne wszystkie.
QUESTIONS = [
    {"pytanie": "Gdzie urodziła się Wisława Szymborska?", "odpowiedz": ["Prowent"], "typ": "jedna"},
    {"pytanie": "Gdzie urodził się Władysław Reymont?", "odpowiedz": ["Kobiele Wielkie"], "typ": "jedna"},
    {"pytanie": "Gdzie urodził się Tadeusz Reichstein?", "odpowiedz": ["Włocławek"], "typ": "jedna"},
    {"pytanie": "Gdzie urodził się Isaac Bashevis Singer?", "odpowiedz": ["Leoncin", "Radzymin"], "typ": "jedna"},
    {"pytanie": "W którym roku Olga Tokarczuk otrzymała Nagrodę Nobla?", "odpowiedz": ["2018"], "typ": "jedna"},
    {"pytanie": "W którym roku Władysław Reymont otrzymał Nagrodę Nobla?", "odpowiedz": ["1924"], "typ": "jedna"},
    {"pytanie": "W którym roku Tadeusz Reichstein otrzymał Nagrodę Nobla?", "odpowiedz": ["1950"], "typ": "jedna"},
    {"pytanie": "Na jakiej uczelni studiowała Wisława Szymborska?", "odpowiedz": ["Uniwersytet Jagielloński"], "typ": "jedna"},
    {"pytanie": "Kto otrzymał Pokojową Nagrodę Nobla?", "odpowiedz": ["Lech Wałęsa", "Joseph Rotblat"], "typ": "lista"},
    {"pytanie": "Kto otrzymał Nagrodę Nobla w dziedzinie chemii?",
     "odpowiedz": ["Maria Skłodowska-Curie", "Fritz Haber", "Walther Nernst", "Friedrich Bergius", "Kurt Alder"], "typ": "lista"},
    {"pytanie": "Którzy laureaci urodzili się w województwie dolnośląskim?",
     "odpowiedz": ["Fritz Haber", "Max Born", "Reinhard Selten", "Gerhart Hauptmann", "Friedrich Bergius", "Paul Ehrlich"], "typ": "lista"},
    {"pytanie": "Kto otrzymał Nagrodę Nobla dwukrotnie?", "odpowiedz": ["Maria Skłodowska-Curie"], "typ": "jedna"},
]

PL = str.maketrans("ąćęłńóśźż", "acelnoszz")


def stems(text, n=5):
    """Zbiór prefiksów słów (bez polskich znaków): prymitywny, ale odporny na odmianę stemmer."""
    return {w[:n] for w in re.findall(r"\w+", text.lower().translate(PL)) if len(w) >= 3}


def key_stem(item):
    return max(re.findall(r"\w+", item.lower().translate(PL)), key=len)[:5]


def score(answer, gold, typ):
    """Zwraca (trafna, odsetek pokrytych elementów wzorca)."""
    st = stems(answer)
    hits = [key_stem(x) in st for x in gold]
    frac = float(np.mean(hits))
    return (frac > 0 if typ == "jedna" else frac == 1.0), frac

In [ ]:
# Zapisz TBox w Turtle: wdt:P131 jako owl:TransitiveProperty oraz ex:urodzonyNaTerenie
# zdefiniowane łańcuchem własności (wdt:P19, wdt:P131); wdt:P19 jako podwłasność ex:urodzonyNaTerenie.
# Skopiuj graf, dołącz TBox i wykonaj owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(...).
# Porównaj liczbę trójek przed i po oraz wynik dwóch zapytań o osoby urodzone w województwie dolnośląskim:
# ze ścieżką wdt:P19/wdt:P131* na grafie bez wnioskowania i z ex:urodzonyNaTerenie po wnioskowaniu.

In [ ]:
# Zbuduj system pytanie -> szablon -> zapytanie do grafu:
# 1) funkcja resolve(fragment, węzły) dopasowująca encję po rdzeniach słów (funkcja stems),
# 2) lista szablonów (wyrażenie regularne, funkcja odpowiedzi) dla sześciu typów pytań z QUESTIONS,
# 3) każda odpowiedź zwraca listę etykiet i uzasadnienie (ścieżkę trójek w grafie).
# Dla pytań o obszar użyj grafu po wnioskowaniu i najkrótszej ścieżki wdt:P19/wdt:P131 w networkx.
# Policz trafność na QUESTIONS funkcją score.

In [ ]:
# Pokaż uzasadnienia dla dwóch pytań oraz odpowiedzi na kilka pytań spoza zestawu
# (np. o Wrocław, o Austro-Węgry, o Kraków i pytanie, którego żaden szablon nie obsługuje).

(uzupełnij)

## Zadanie 4: Wyszukiwanie semantyczne faktów

In [ ]:
# Zamień trójki grafu na zdania po polsku (miejsce urodzenia, nagroda z rokiem, uczelnia, wdt:P131),
# uwzględniając rodzaj gramatyczny z ex:plec. Wypisz liczbę faktów i kilka przykładów.

In [ ]:
# Zakoduj fakty i pytania modelem paraphrase-multilingual-MiniLM-L12-v2 (znormalizowane wektory, CPU).
# Dla porównania zbuduj TF-IDF na n-gramach znakowych (char_wb, 3 do 5).
# Dla k = 1, 3, 5, 10 policz średnie pokrycie odpowiedzi: jaka część elementów wzorca pojawia się w k najbliższych faktach.

In [ ]:
# Pokaż 4 najbliższe fakty (z podobieństwem) dla trzech pytań: o miejsce urodzenia, o rok nagrody i o województwo.

(uzupełnij)

## Zadanie 5: Graf wiedzy a model językowy

In [ ]:
# Wczytaj Qwen/Qwen2.5-0.5B-Instruct (AutoTokenizer, AutoModelForCausalLM, dtype=torch.float32, CPU).
# Napisz funkcję ask(question, context=None) z komunikatem systemowym, szablonem czatu (apply_chat_template),
# generowaniem zachłannym (do_sample=False, max_new_tokens=40) i pomiarem czasu.

In [ ]:
# Zadaj wszystkie pytania z QUESTIONS w czterech wariantach: bez kontekstu, z 5 najbliższymi faktami
# z zadania 4, z surowymi ścieżkami uzasadnień z zadania 3 oraz z tymi samymi ścieżkami zapisanymi jako zdania.
# Dla każdej odpowiedzi zapisz trafność (score), czy to odmowa, czas; halucynacja = nietrafna i nie odmowa.

In [ ]:
# Zbuduj tabelę zbiorczą: trafność, średnie pokrycie, liczba halucynacji i odmów, średni czas,
# z dodatkowym wierszem dla systemu szablonów z zadania 3. Pokaż wszystkie odpowiedzi obok siebie.

(uzupełnij)

## Zadanie 6: Pytania kontrolne

(uzupełnij odpowiedzi na pytania kontrolne z instrukcji)